In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

PROJECT_ROOT = Path(r'C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump')
INPUT_FILE = PROJECT_ROOT / 'data' / 'processed' / 'integrated' / 'crop_soil_climate_complete_corrected_2013_2025.csv'
OUTPUT_FILE = PROJECT_ROOT / 'data' / 'processed' / 'baseline' / 'crop_soil_climate_with_10yr_baseline_2013_2025.csv'
AUDIT_FILE = PROJECT_ROOT / 'data' / 'processed' / 'baseline' / 'historical_10yr_baseline_audit.csv'
WINDOW_YEARS = 10

print('Input :', INPUT_FILE)
print('Output:', OUTPUT_FILE)
print('Audit :', AUDIT_FILE)

Input : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_corrected_2013_2025.csv
Output: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\baseline\crop_soil_climate_with_10yr_baseline_2013_2025.csv
Audit : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\baseline\historical_10yr_baseline_audit.csv


In [12]:
df = pd.read_csv(INPUT_FILE)
print('Shape:', df.shape)

required = ['year','state','district','crop','soil_type','yield_kg_ha']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['year'] = pd.to_numeric(df['year'], errors='raise').astype(int)
df['yield_kg_ha'] = pd.to_numeric(df['yield_kg_ha'], errors='coerce')

expected_years = set(range(2013, 2025))
assert set(df['year'].unique()) == expected_years
assert len(df) == 67826
assert (df['yield_kg_ha'].dropna() >= 0).all()

print('Years:', sorted(df['year'].unique()))
print('Missing yield:', int(df['yield_kg_ha'].isna().sum()))
print('PASS: input validation')

Shape: (67826, 53)
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Missing yield: 2355
PASS: input validation


In [13]:
#Define the historical baseline key
BASELINE_KEYS = ['state','district','crop','soil_type']
KEY_YEAR = BASELINE_KEYS + ['year']

for col in BASELINE_KEYS:
    df[col] = df[col].astype('string').str.strip()

dup_count = int(df.duplicated(KEY_YEAR).sum())
print('Duplicate key-year rows:', dup_count)
if dup_count:
    display(df[df.duplicated(KEY_YEAR, keep=False)][KEY_YEAR + ['yield_kg_ha']].head(20))
    raise ValueError('Duplicate State+District+Crop+Soil Type+Year records found.')

print('PASS: baseline key is unique within each year')

Duplicate key-year rows: 0
PASS: baseline key is unique within each year


In [14]:
# Sort data
df = df.sort_values(BASELINE_KEYS + ['year']).reset_index(drop=True)

baseline_col = 'historical_10yr_baseline_yield_kg_ha'
count_col = 'historical_baseline_year_count'
start_col = 'historical_baseline_start_year'
end_col = 'historical_baseline_end_year'

# Keep the original row order/index
df['_original_row'] = np.arange(len(df))

# Calculate baseline row-by-row using State + District + Crop + Soil Type
baseline_values = []
baseline_counts = []
baseline_starts = []
baseline_ends = []

for idx, row in df.iterrows():

    current_year = row['year']

    # Match the same baseline group
    mask = (
        (df['state'] == row['state']) &
        (df['district'] == row['district']) &
        (df['crop'] == row['crop']) &
        (df['soil_type'] == row['soil_type']) &
        (df['year'] < current_year) &
        (df['year'] >= current_year - WINDOW_YEARS)
    )

    historical = df.loc[mask, ['year', 'yield_kg_ha']].copy()

    # Ignore missing historical yields
    historical = historical.dropna(subset=['yield_kg_ha'])

    if len(historical) > 0:
        baseline_values.append(historical['yield_kg_ha'].mean())
        baseline_counts.append(len(historical))
        baseline_starts.append(historical['year'].min())
        baseline_ends.append(historical['year'].max())
    else:
        baseline_values.append(np.nan)
        baseline_counts.append(0)
        baseline_starts.append(np.nan)
        baseline_ends.append(np.nan)


df[baseline_col] = baseline_values
df[count_col] = baseline_counts
df[start_col] = baseline_starts
df[end_col] = baseline_ends

df = df.drop(columns=['_original_row'])

print('Baseline calculation completed.')
print('Shape:', df.shape)
print('Required key columns preserved:',
      all(c in df.columns for c in BASELINE_KEYS))

Baseline calculation completed.
Shape: (67826, 57)
Required key columns preserved: True


In [15]:
#Diagnostics
print('Baseline availability:')
print(df[baseline_col].notna().value_counts())

print('\nHistorical count distribution:')
print(df[count_col].describe())

print('\nBaseline statistics:')
print(df[baseline_col].describe())

print('\nFull 10-year history:', int((df[count_col] == 10).sum()))
print('Partial history:', int(((df[count_col] > 0) & (df[count_col] < 10)).sum()))
print('No history:', int((df[count_col] == 0).sum()))

Baseline availability:
historical_10yr_baseline_yield_kg_ha
True     59657
False     8169
Name: count, dtype: int64

Historical count distribution:
count    67826.000000
mean         4.771061
std          3.318945
min          0.000000
25%          2.000000
50%          5.000000
75%          8.000000
max         10.000000
Name: historical_baseline_year_count, dtype: float64

Baseline statistics:
count     59657.000000
mean       6894.493617
std       18791.292894
min           0.000000
25%         883.900000
50%        1378.000000
75%        2473.250000
max      220618.000000
Name: historical_10yr_baseline_yield_kg_ha, dtype: float64

Full 10-year history: 8006
Partial history: 51651
No history: 8169


In [16]:
#Leakage validation
with_history = df[df[count_col] > 0]

assert (with_history[start_col] < with_history['year']).all()
assert (with_history[end_col] < with_history['year']).all()
assert (df[count_col] >= 0).all()
assert (df[count_col] <= WINDOW_YEARS).all()
assert (df.duplicated(KEY_YEAR).sum() == 0)

print('PASS: no current-year yield is used in the historical baseline.')

PASS: no current-year yield is used in the historical baseline.


In [17]:
#Optional baseline comparison feature
df['yield_vs_historical_baseline_ratio'] = np.where(
    df[baseline_col] > 0,
    df['yield_kg_ha'] / df[baseline_col],
    np.nan
)
print(df['yield_vs_historical_baseline_ratio'].describe())

count    58185.000000
mean         1.140106
std          5.348166
min          0.000000
25%          0.920090
50%          1.042880
75%          1.210322
max       1200.643599
Name: yield_vs_historical_baseline_ratio, dtype: float64


In [18]:
#Final audit and save
audit = pd.DataFrame({
    'metric': [
        'input_rows','output_rows','window_years','baseline_rows_available',
        'baseline_rows_unavailable','full_10_year_history_rows',
        'partial_history_rows','duplicate_key_year_rows','minimum_year','maximum_year'
    ],
    'value': [
        67826, len(df), WINDOW_YEARS,
        int(df[baseline_col].notna().sum()),
        int(df[baseline_col].isna().sum()),
        int((df[count_col] == WINDOW_YEARS).sum()),
        int(((df[count_col] > 0) & (df[count_col] < WINDOW_YEARS)).sum()),
        int(df.duplicated(KEY_YEAR).sum()),
        int(df['year'].min()), int(df['year'].max())
    ]
})

display(audit)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)
audit.to_csv(AUDIT_FILE, index=False)

print('\nSaved dataset:', OUTPUT_FILE)
print('Saved audit  :', AUDIT_FILE)
print('Final shape  :', df.shape)

,metric,value
0,input_rows,67826
1,output_rows,67826
2,window_years,10
3,baseline_rows_available,59657
4,baseline_rows_unavailable,8169
5,full_10_year_history_rows,8006
6,partial_history_rows,51651
7,duplicate_key_year_rows,0
8,minimum_year,2013
9,maximum_year,2024



Saved dataset: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\baseline\crop_soil_climate_with_10yr_baseline_2013_2025.csv
Saved audit  : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\baseline\historical_10yr_baseline_audit.csv
Final shape  : (67826, 58)
